In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch

BASE_PATH = Path(
    "/content/drive/MyDrive/ECs Master Folder/Research/"
    "Kidney Thermal Property MFNN/Data"
)

# nested dictionaries
# outer dictionary (text as key) -> dictionaries as the value
# that dictionary value has key-value pairs
EXPERIMENTS = {
    "porcine_conductivity": {
        "file": "porcine_raw.csv",
        "target": "conductivity_mean_W_mK",
        "uncertainty": "conductivity_EU95_W_mK",
    },
    "porcine_diffusivity": {
        "file": "porcine_raw.csv",
        "target": "diffusivity_mean_mm2_s",
        "uncertainty": "diffusivity_EU95_mm2_s",
    },
    "bovine_conductivity": {
        "file": "bovine_raw.csv",
        "target": "conductivity_mean_W_mK",
        "uncertainty": "conductivity_EU95_W_mK"
    },
    "bovine_diffusivity": {
        "file": "bovine_raw.csv",
        "target": "diffusivity_mean_mm2_s",
        "uncertainty": "diffusivity_EU95_mm2_s",
    },
}

TEMP_COL = "temperature_C"
HF_SIZES = [3, 5, 7, 9]
SEEDS = range(20)

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
def load_experiment(config): # config = one inner dictionary (like porcine conductivity)
  df = pd.read_csv(BASE_PATH / config["file"]) # opens csv file

  required_columns = [
      TEMP_COL,
      config["target"],
      config["uncertainty"]
  ]

  missing_columns = []

  # adds column
  for column in required_columns:
    if column not in df.columns:
        missing_columns.append(column) # stores a variable w/ missing column names

  if missing_columns: # non-empty list -> True, empty list -> False
    raise ValueError(
        f"Missing columns: {missing_columns}\n"
        f"Available columns: {list(df.columns)}"
    )

  return (
      df[required_columns]
      .dropna() # deletes rows w/ missing values
      .sort_values(TEMP_COL) # low -> high
      .reset_index(drop=True) # resets row indices
  )